<a href="https://colab.research.google.com/github/santiiis/proyectobigdata/blob/main/Proyecto_Final_BigData_Desercion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Proyecto Integrador: Predicción de deserción estudiantil**

**Integrantes:** Lander González, Erick Morales  
**Tema:** Predicción de deserción estudiantil (Grupo 7)  
**Asignatura:** Prácticas y Herramientas de Big Data  




### 1. Configuración de Kaggle y Carga de Datos Real OULAD
El dataset se descarga automáticamente utilizando la API de Kaggle configurada con el token del equipo.

In [1]:
import os

# Configuración de credenciales de Kaggle
os.environ['KAGGLE_USERNAME'] = "anlgrbz"
os.environ['KAGGLE_KEY'] = "KGAT_04a9d6e205159a309180f748b0052daa"

# Descarga y descompresión del dataset
if not os.path.exists('/content/oulad_data'):
    !kaggle datasets download -d anlgrbz/student-demographics-online-education-dataoulad -p /content/oulad_data --unzip

print("Dataset OULAD descargado correctamente en /content/oulad_data")

Dataset OULAD descargado correctamente en /content/oulad_data


### 2. Carga de Datos con Claves Compuestas
Para evitar la duplicidad de registros, las uniones se realizan mediante `id_student`, `code_module` y `code_presentation`.

###  Inicialización de Sesión Distribuida en Apache Spark y Carga de Datos OULAD

In [2]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y SESIÓN DISTRIBUIDA DE SPARK
# ==============================================================================
!pip install -q pyspark mlflow

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.spark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, mean, stddev, abs as spark_abs
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Iniciar Spark Session Distribuida
spark = SparkSession.builder \
    .appName("Prediccion_Desercion_OULAD_BigData") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Sesión Spark iniciada con éxito. Versión: {spark.version}")

Sesión Spark iniciada con éxito. Versión: 4.0.3


In [3]:
# Cargamos las tablas base
student_info = spark.read.csv("oulad_data/studentInfo.csv", header=True, inferSchema=True)
student_vle = spark.read.csv("oulad_data/studentVle.csv", header=True, inferSchema=True)
student_reg = spark.read.csv("oulad_data/studentRegistration.csv", header=True, inferSchema=True)

# Agregamos clics por estudiante, módulo y presentación
sum_click_df = student_vle.groupBy("id_student", "code_module", "code_presentation").sum("sum_click") \
    .withColumnRenamed("sum(sum_click)", "sum_click")

# Join usando claves compuestas
df_spark_raw = student_info.join(sum_click_df, ["id_student", "code_module", "code_presentation"], "left") \
    .join(student_reg.select("id_student", "code_module", "code_presentation", "date_registration"),
          ["id_student", "code_module", "code_presentation"], "left")

# Imputación corregida según recomendaciones
df_spark_raw = df_spark_raw.withColumn(
    "label",
    when(col("final_result") == "Withdrawn", 1).otherwise(0)
).fillna({
    'sum_click': 0,
    'num_of_prev_attempts': 0,
    'studied_credits': 60
})

df_spark_raw.show(5)

+----------+-----------+-----------------+------+--------------------+--------------------+--------+--------+--------------------+---------------+----------+------------+---------+-----------------+-----+
|id_student|code_module|code_presentation|gender|              region|   highest_education|imd_band|age_band|num_of_prev_attempts|studied_credits|disability|final_result|sum_click|date_registration|label|
+----------+-----------+-----------------+------+--------------------+--------------------+--------+--------+--------------------+---------------+----------+------------+---------+-----------------+-----+
|     11391|        AAA|            2013J|     M| East Anglian Region|    HE Qualification| 90-100%|    55<=|                   0|            240|         N|        Pass|      934|             -159|    0|
|     31604|        AAA|            2013J|     F|   South East Region|A Level or Equiva...|  50-60%|   35-55|                   0|             60|         N|        Pass|     2158|

### 2. Pipeline ETL Distribuido y Persistencia en Formato Columnar Parquet Particionado (Fase II)
**Transformaciones aplicadas:**
1. Filtrado de registros fuera de rango temporal (-150 a +30 días).
2. Imputación de valores nulos en telemetría de clics (`sum_click`).
3. Imputación de historial académico (`num_of_prev_attempts`).
4. Persistencia en formato Parquet particionado por `code_module` para optimizar I/O.

In [4]:
# ETL Sincronizado: Rango -150 a +30 días
df_filtered = df_spark_raw.filter((col("date_registration") >= -150) & (col("date_registration") <= 30))

# Validación de impacto (Requerido por rúbrica)
count_raw = df_spark_raw.count()
count_filtered = df_filtered.count()
print("--- VALIDACIÓN DE IMPACTO ETL ---")
print(f"Registros iniciales:      {count_raw:,}")
print(f"Registros tras filtrado:  {count_filtered:,}")
print(f"Registros descartados:    {count_raw - count_filtered:,}")

# Persistencia
parquet_path = "oulad_parquet_lake"
df_filtered.write.mode("overwrite").partitionBy("code_module").parquet(parquet_path)

# Split
df_lake = spark.read.parquet(parquet_path)
train_data, test_data = df_lake.randomSplit([0.8, 0.2], seed=42)
print(f"\nDistribución: Train: {train_data.count():,} | Test: {test_data.count():,}")

--- VALIDACIÓN DE IMPACTO ETL ---
Registros iniciales:      32,593
Registros tras filtrado:  30,320
Registros descartados:    2,273

Distribución: Train: 24,298 | Test: 6,022


### 3. Modelado con Spark MLlib, ParamGrid, CrossValidator y MLflow Tracking (TA-4.1 y TA-4.2)
Se evalúan y registran 3 modelos formales en MLflow:
1. **Regresión Logística**
2. **Gradient-Boosted Trees (GBT)**
3. **Random Forest Classifier**

In [5]:
mlflow.set_experiment("Proyecto_OULAD_Optimizado")

feature_cols = ['sum_click', 'studied_credits', 'num_of_prev_attempts', 'date_registration']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

models_config = [
    ("LogisticRegression", LogisticRegression(featuresCol="scaledFeatures", labelCol="label"),
     ParamGridBuilder().addGrid(LogisticRegression.regParam, [0.01, 0.1]).build()),
    ("GBTClassifier", GBTClassifier(featuresCol="features", labelCol="label", seed=42),
     ParamGridBuilder().addGrid(GBTClassifier.maxDepth, [3, 5]).build()),
    ("RandomForest", RandomForestClassifier(featuresCol="features", labelCol="label", seed=42),
     ParamGridBuilder().addGrid(RandomForestClassifier.numTrees, [20, 50]).build())
]

best_overall_auc = 0.0
best_model_pipeline = None
final_predictions = None

for name, model, grid in models_config:
    with mlflow.start_run(run_name=name):
        stages = [assembler, scaler, model] if name == "LogisticRegression" else [assembler, model]
        pipeline = Pipeline(stages=stages)

        cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator_auc, numFolds=3)
        cv_model = cv.fit(train_data)

        # --- REGISTRO DE PARÁMETROS GANADORES ---
        best_idx = int(np.argmax(cv_model.avgMetrics))
        best_params = cv_model.getEstimatorParamMaps()[best_idx]
        for param, value in best_params.items():
            mlflow.log_param(param.name, value)

        preds = cv_model.transform(test_data)
        auc_val = evaluator_auc.evaluate(preds)
        f1_val = evaluator_f1.evaluate(preds)

        mlflow.log_metric("auc", auc_val)
        mlflow.log_metric("f1_score", f1_val)
        print(f"[{name}] AUC: {auc_val:.4f} | F1: {f1_val:.4f}")

        if auc_val > best_overall_auc:
            best_overall_auc = auc_val
            best_model_pipeline = cv_model.bestModel
            final_predictions = preds

print(f"\nMejor modelo detectado: {best_model_pipeline.stages[-1].__class__.__name__} con AUC: {best_overall_auc:.4f}")

[LogisticRegression] AUC: 0.8349 | F1: 0.7889
[GBTClassifier] AUC: 0.8480 | F1: 0.7994
[RandomForest] AUC: 0.8205 | F1: 0.7981

Mejor modelo detectado: GBTClassificationModel con AUC: 0.8480


### 4. Evaluación de Resultados, Feature Importance y Detección de Anomalías (TA-4.3)

In [6]:
# ==============================================================================
# 4. EVALUACIÓN FINAL: IMPORTANCIA, MATRIZ Y ANOMALÍAS
# ==============================================================================

# 1. Extraer el objeto del modelo final
best_model_stage = best_model_pipeline.stages[-1]
print(f"--- EVALUACIÓN DEL MEJOR MODELO: {best_model_stage.__class__.__name__} ---")

# 2. Mostrar importancia de variables si es un modelo basado en árboles
if hasattr(best_model_stage, "featureImportances"):
    importances = best_model_stage.featureImportances.toArray()
    print("\n--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS ---")
    for col_name, imp in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
        print(f" • {col_name:20s}: {imp * 100:.2f}%")
elif hasattr(best_model_stage, "coefficients"):
    print("\nNota: El modelo es lineal. Los coeficientes son:")
    for col_name, coef in zip(feature_cols, best_model_stage.coefficients.toArray()):
        print(f" • {col_name:20s}: {coef:.4f}")

# 3. Matriz de Confusión manual
tp = final_predictions.filter((col("label") == 1) & (col("prediction") == 1.0)).count()
tn = final_predictions.filter((col("label") == 0) & (col("prediction") == 0.0)).count()
fp = final_predictions.filter((col("label") == 0) & (col("prediction") == 1.0)).count()
fn = final_predictions.filter((col("label") == 1) & (col("prediction") == 0.0)).count()

print("\n--- MATRIZ DE CONFUSIÓN (CONJUNTO DE PRUEBA) ---")
print(f"Verdaderos Positivos (TP): {tp:,}")
print(f"Falsos Positivos (FP):     {fp:,}")
print(f"Verdaderos Negativos (TN): {tn:,}")
print(f"Falsos Negativos (FN):     {fn:,}")

# 4. Análisis de Anomalías
stats = df_filtered.select(mean("sum_click"), stddev("sum_click")).collect()[0]
m_clicks, s_clicks = stats[0], stats[1]

anomalies = df_filtered.withColumn("z_score", spark_abs((col("sum_click") - m_clicks) / s_clicks)) \
                      .filter(col("z_score") > 3.0)

anomalies_count = anomalies.count()
total_count = df_filtered.count()
print(f"\n--- ANÁLISIS DE ANOMALÍAS ---")
print(f"Registros atípicos en clicks (|Z| > 3.0): {anomalies_count:,} ({anomalies_count/total_count*100:.2f}% del total)")
print("\n=====================================================================")
print("PROCESO COMPLETADO Y VALIDADO")
print("=====================================================================")

--- EVALUACIÓN DEL MEJOR MODELO: GBTClassificationModel ---

--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS ---
 • sum_click           : 74.28%
 • studied_credits     : 12.80%
 • date_registration   : 9.02%
 • num_of_prev_attempts: 3.90%

--- MATRIZ DE CONFUSIÓN (CONJUNTO DE PRUEBA) ---
Verdaderos Positivos (TP): 941
Falsos Positivos (FP):     331
Verdaderos Negativos (TN): 3,934
Falsos Negativos (FN):     816

--- ANÁLISIS DE ANOMALÍAS ---
Registros atípicos en clicks (|Z| > 3.0): 615 (2.03% del total)

PROCESO COMPLETADO Y VALIDADO
